# Stage III. Out-of-fold predictions and threshold calibration

Every development image is scored by the one fold model that did not train on it. These
out-of-fold (OOF) predictions give the cross-validation table of the paper and are the only
data used to fit the four decision thresholds.

**Decode.** Output Grade 5 if `p_g < t_g`; otherwise `1 + sum_k prod_{j<=k} 1[p_j > t_j]`
(rank-consistent CORN rank, stops at the first failed threshold). The thresholds are fitted
by coordinate ascent on pooled OOF severity QWK.

**Steps.**
1. Rebuild the development table and folds (identical to notebook 01).
2. Check the configuration stored in each checkpoint.
3. OOF inference: `oof_corn.csv` and the per-fold table.
4. Fit `(t1, t2, t3), t_g` on OOF and report the gain.

## 0. Configuration

In [ ]:
from pathlib import Path

# Development pool (18,570 DR-positive images, grades 1-5) and its Stage II masks.
#   data/stage3/dev_grades.csv              columns: image, grade (1..5)
#   data/stage3/dev_images/<stem>.png|jpg
#   data/stage3/dev_masks/<lesion>/<stem><suffix>.png   lesion in {ma, he, ex, cws, vessel};
#       suffix: ma "_mask", he "_he_mask", ex "_ex_mask", cws "_cws_mask", vessel "_mask"
DATA_ROOT = Path("data")
DEV_CSV = DATA_ROOT / "stage3" / "dev_grades.csv"
DEV_IMG_DIR = DATA_ROOT / "stage3" / "dev_images"
DEV_MASK_ROOT = DATA_ROOT / "stage3" / "dev_masks"

OUT_ROOT = Path("outputs")
CKPT_DIR = OUT_ROOT / "stage3" / "checkpoints"      # best_fold{0..4}.pt
EVAL_DIR = OUT_ROOT / "stage3" / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

import json
import torch
import pandas as pd
from hierarchiretina.stage3.engine import LGDRGConfig

cfg = LGDRGConfig(out_dir=str(CKPT_DIR))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Development table and folds

In [ ]:
from hierarchiretina.stage3.data import prepare_development_set

df = prepare_development_set(DEV_CSV, DEV_IMG_DIR, DEV_MASK_ROOT, cfg.n_folds, cfg.seed)

## 2. Checkpoint provenance

Each checkpoint stores the configuration it was trained with. This is the authoritative record
of, for example, `grad_accum` and `batch_size` for the released models.

In [ ]:
keys = ["backbone", "img_size", "batch_size", "grad_accum", "lr", "weight_decay", "warmup_epochs",
        "ema_decay", "epochs", "early_stop_patience"]
rows = []
for k in range(cfg.n_folds):
    ck = torch.load(CKPT_DIR / f"best_fold{k}.pt", map_location="cpu", weights_only=False)
    rows.append({"fold": k, "best_epoch": ck["epoch"], "best_macro_auc": ck["best"],
                 **{c: ck["cfg"].get(c) for c in keys}})
    del ck
pd.DataFrame(rows)

## 3. Out-of-fold inference

Writes `oof_corn.csv` (image_id, fold, true_grade, y, pred_idx, pred_grade, p_grade_1..5) and
`oof_per_fold_metrics.csv`. Severity QWK here uses the paper definition (true Grade 5 removed,
predicted Grade 5 mapped to 4) with the default 0.5 decode.

In [ ]:
from hierarchiretina.stage3.engine import oof_predictions

OOF_CSV = EVAL_DIR / "oof_corn.csv"
if OOF_CSV.exists():
    oof = pd.read_csv(OOF_CSV)
    per_fold = pd.read_csv(EVAL_DIR / "oof_per_fold_metrics.csv")
else:
    oof, per_fold = oof_predictions(cfg, df, CKPT_DIR, device=DEVICE)
    oof.to_csv(OOF_CSV, index=False)
    per_fold.to_csv(EVAL_DIR / "oof_per_fold_metrics.csv", index=False)

### Table: five-fold cross-validation (single model per image)

In [ ]:
tab = per_fold.rename(columns={"auc": "macro_auc", "g5_auc": "G5_auc", "qwk_sev": "QWK",
                               "acc": "accuracy", "f1": "macro_F1"}).set_index("fold")
tab.loc["mean"] = tab.mean()
tab.loc["sd"] = tab.iloc[:5].std()      # sample SD over folds
tab.round(3)

## 4. Threshold calibration on OOF

Coordinate ascent from `(0.5, 0.5, 0.5), 0.5`: each round sweeps `t1, t2, t3` over
0.05-0.95 (step 0.01) and then `t_g` over 0.10-0.90 (step 0.02), keeping a value only if it
improves OOF QWK. The released OOF file gives `t = (0.38, 0.60, 0.19)` and `t_g = 0.10`.

In [ ]:
from hierarchiretina.stage3.calibration import calibrate_thresholds
from hierarchiretina.stage3.cascade import oof_calibration_summary, PC

t, tg, best, start = calibrate_thresholds(oof.true_grade.values, oof[PC].values)
calib, oof_per_grade = oof_calibration_summary(oof, t, tg, best, start)
json.dump(calib, open(EVAL_DIR / "corn_thresholds.json", "w"), indent=2)
print(json.dumps(calib, indent=2))

### Per-grade effect of the tuned decode (OOF)

In [ ]:
oof_per_grade.round(3)